In [2]:
# load the document
from langchain_community.document_loaders import PyPDFLoader
from transformers.testing_utils import preprocess_string

file_path = "rag-doc.pdf"
loader = PyPDFLoader(file_path)

document = loader.load()

document[0]

Document(metadata={'producer': 'pdfTeX, Version 3.141592653-2.6-1.40.25 (TeX Live 2023) kpathsea version 6.3.5', 'creator': 'LaTeX with acmart 2023/03/30 v1.90 Typesetting articles for the Association for Computing Machinery and hyperref 2023-04-22 v7.00x Hypertext links for LaTeX', 'creationdate': '2024-08-12T00:25:21+00:00', 'moddate': '2024-08-12T00:25:21+00:00', 'ptex.fullbanner': 'This is pdfTeX, Version 3.141592653-2.6-1.40.25 (TeX Live 2023) kpathsea version 6.3.5', 'subject': '', 'title': 'HybridRAG: Integrating Knowledge Graphs and Vector Retrieval Augmented Generation for Efficient Information Extraction', 'trapped': '/False', 'source': 'rag-doc.pdf', 'total_pages': 9, 'page': 0, 'page_label': '1'}, page_content='HybridRAG: Integrating Knowledge Graphs and Vector Retrieval\nAugmented Generation for Efficient Information Extraction\nBhaskarjit Sarmah\nbhaskarjit.sarmah@blackrock.com\nBlackRock, Inc.\nGurugram, India\nBenika Hall\nbhall@nvidia.com\nNVIDIA\nSanta Clara, CA, USA\

In [3]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=100,
    chunk_overlap=0
)
chunks = text_splitter.split_documents(document)

In [4]:
chunks[0].metadata

{'producer': 'pdfTeX, Version 3.141592653-2.6-1.40.25 (TeX Live 2023) kpathsea version 6.3.5',
 'creator': 'LaTeX with acmart 2023/03/30 v1.90 Typesetting articles for the Association for Computing Machinery and hyperref 2023-04-22 v7.00x Hypertext links for LaTeX',
 'creationdate': '2024-08-12T00:25:21+00:00',
 'moddate': '2024-08-12T00:25:21+00:00',
 'ptex.fullbanner': 'This is pdfTeX, Version 3.141592653-2.6-1.40.25 (TeX Live 2023) kpathsea version 6.3.5',
 'subject': '',
 'title': 'HybridRAG: Integrating Knowledge Graphs and Vector Retrieval Augmented Generation for Efficient Information Extraction',
 'trapped': '/False',
 'source': 'rag-doc.pdf',
 'total_pages': 9,
 'page': 0,
 'page_label': '1'}

In [5]:
# perform Embedding
from langchain_openai import OpenAIEmbeddings
from dotenv import load_dotenv

load_dotenv()

embedding_model = OpenAIEmbeddings()
embedding_model

OpenAIEmbeddings(client=<openai.resources.embeddings.Embeddings object at 0x152871be0>, async_client=<openai.resources.embeddings.AsyncEmbeddings object at 0x152872510>, model='text-embedding-ada-002', dimensions=None, deployment='text-embedding-ada-002', openai_api_version=None, openai_api_base=None, openai_api_type=None, openai_proxy=None, embedding_ctx_length=8191, openai_api_key=SecretStr('**********'), openai_organization=None, allowed_special=None, disallowed_special=None, chunk_size=1000, max_retries=2, request_timeout=None, headers=None, tiktoken_enabled=True, tiktoken_model_name=None, show_progress_bar=False, model_kwargs={}, skip_empty=False, default_headers=None, default_query=None, retry_min_seconds=4, retry_max_seconds=20, http_client=None, http_async_client=None, check_embedding_ctx_length=True)

In [27]:
!pip install faiss-cpu

In [6]:
# embedd and store into vector space, for testing local vector space will use
from langchain_community.vectorstores import FAISS

In [7]:
vectorstore = FAISS.from_documents(chunks, embedding_model)

In [8]:
# Save the index locally (Optional - so you don't have to re-embed later)
vectorstore.save_local("faiss_index")

In [11]:
# now do the sematic search
query = "What is retrieval augmented generation?"

dense_results = vectorstore.similarity_search(query, k=5)
for doc in dense_results:
    print(f"doc : {doc.page_content[:200], len(doc.page_content)}")

doc : ('trieval Augmented Generation (RAG) (referred to as VectorRAG', 60)
doc : ('Retrieval-Augmented Generation (RAG) techniques [9], which aim', 62)
doc : ('Rocktäschel, et al. Retrieval-augmented generation for knowledge-intensive nlp', 78)
doc : ('awei Sun, and Haofen Wang. Retrieval-augmented generation for large language', 76)
doc : ('gas: Automated evaluation of retrieval augmented generation. arXiv preprint\narXiv:2309.15217, 2023.', 99)


In [12]:
# Sparse Search

from rank_bm25 import BM25Okapi

corpus = [doc.page_content for doc in chunks]
tokenized_corpus = [doc.split() for doc in corpus]

bm25 = BM25Okapi(tokenized_corpus)


In [13]:
bm25

In [15]:
import numpy as np

tokenized_query = query.split()

bm25_scores = bm25.get_scores(tokenized_query)

top_k = 5
ranked_indices = np.argsort(bm25_scores)[::-1][:top_k]

sparsed_result = [chunks[i] for i in ranked_indices]

for doc in sparsed_result:
    print(f"doc : {doc.page_content[:200], len(doc.page_content)}")


doc : ('gas: Automated evaluation of retrieval augmented generation. arXiv preprint\narXiv:2309.15217, 2023.', 99)
doc : ('augmented generation. arXiv preprint arXiv:2402.05131, 2024.', 60)
doc : ('augmented generation for open-domain question answering. In Proceedings', 71)
doc : ('for a given question. This retrieval process is fine-tuned with spe-', 68)
doc : ('retrieval techniques. VectorRAG (the traditional RAG techniques', 63)
